# 训练 PM2.5 预测模型

[English](model_training.ipynb) | 简体中文

从[模型包](source/models/README.md)发现预测类，用按时间划分的训练集拟合各配置。
用验证集比较模型声明的预算，为每个配置保存一个 checkpoint。
[测试 Notebook](model_test_comparison.zh-CN.ipynb)加载这些 checkpoint，
在预留测试集上运行推理。

## 准备输入数据

首次运行时，在项目根目录依次执行以下命令。Notebook 的 Python kernel 使用项目的
[固定依赖](requirements.txt)。

```bash
python -B -m source.prepare
python -B -m source.prepare_recent_weather
python -B -m source.build_experiment_matrices
```

从项目根目录打开 Notebook。重新训练时给 `RUN_DIR` 设置一个新目录，
测试 Notebook 也使用同一个路径。

In [ ]:
import sys

sys.dont_write_bytecode = True

from source.comparison import comparison_configs, train_comparison
from source.experiment_paths import CHECKPOINTS, MATRIX
from source.models import discover_models

TRAIN_SEED = 42
THREADS = 4
RUN_DIR = CHECKPOINTS

models = discover_models()
configs = comparison_configs(TRAIN_SEED)
print("Model packages:", ", ".join(models))
print("Configurations:", len(configs))

## 拟合与验证

原有东区和五区模型用 25,668 个训练起点。新增配置声明自己的输入表示，
以及是否使用包含 59,601 个起点的扩展训练集。紧凑特征用相应训练集计算中位数，
填补参数随模型一同保存。所有配置共用 3,331 个验证起点。

本 Notebook 对每个配置选择验证集 MSE 最小的声明预算，再遵循模型的最终训练容量。
原命令行流程用按时间划分的交叉验证选择预算。

In [ ]:
entries = train_comparison(MATRIX, RUN_DIR, seed=TRAIN_SEED, threads=THREADS)
validation_order = sorted(entries, key=lambda name: entries[name]["validation_mse"])
print("Validation selection:", validation_order[0])